## Electrostatic problem
Electric potential and current density are produced in the coil.

Solve a potential problem to determine current density in wire.
Solve for potential $\Phi$ and current density $j$ on the domain $\Omega_{\text{coil}}$.

\begin{align*}
j & = \sigma \nabla \Phi \\
\operatorname{div} j & = 0
\end{align*}
with electric conductivity $\sigma$.

Port BSs are as follows: 
\begin{align*}
\Phi & = 0  \qquad \qquad \text{on } \Gamma_{\text{out}},  \\
j_n & = \frac{I_{coil}}{|\Gamma_{in}|} \quad \qquad \text{on } \Gamma_{\text{in}},
\end{align*}

and $j_n=0$ else

In [7]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../../meshes/coil_box_named'
output_path = '../../output/case1/case1_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

# for i in range(1, 3):
#     # print(i)
#     mesh.SetMaterial(i, f'{i}')

# for i in range(1, 13):
#     # print(i)
#     mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [8]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629, 12090, ('vacuum', 'wire'), ('CoilIn', 'CoilOut'))

In [9]:
# Define material, coil, and BC parameters

I_coil = 2191.9 # Input electric current [A]
sigma_coil = 62.83185 # Coil's electric conductivity [S/m]

In [10]:
sigma = {"vacuum": 0.0, "wire": sigma_coil}  # Electric conductivity [S/m]
sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])
crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("CoilIn"))

print(f"Coil cross section = {crosssection} m^2.")

fespot = H1(mesh, order=1, definedon=mesh.Materials("wire"), dirichlet="CoilOut")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("CoilIn")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec

gfcurrdens = -sigma_cf*grad(gfphi)

Coil cross section = 7.0710678118656e-05 m^2.


In [11]:
fespot_global = H1(mesh, order=1)
gfphi_global = GridFunction(fespot_global)
gfphi_global.Set(gfphi, definedon=mesh.Materials("wire"))

fescurrden_global = VectorH1(mesh, order=1)
# fescurrden_global = VectorL2(mesh, order=0)
# fescurrden_global = HCurl(mesh, order=1)
# fescurrden_global = HDiv(mesh, order=1)
gfcurrdens_global = GridFunction(fescurrden_global)
gfcurrdens_global.Set(gfcurrdens, definedon=mesh.Materials("wire"))

In [12]:
# vtk = VTKOutput(mesh,coefs=[gfphi],names=["sol"],filename=output_path + "electric_potential",subdivision=0)
# vtk.Do()

res = pv.read(mesh_path + ".msh")
points = res.points
gfphi_out = np.zeros((points.shape[0], 1))
gfcurrdens_out = np.zeros((points.shape[0], 3))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    gfphi_out[i, :] = gfphi_global(point)
    gfcurrdens_out[i, :] = gfcurrdens_global(point)

res["electric_potential"] = gfphi_out
res["current_density"] = gfcurrdens_out

res.save(output_path + ".vtu")